<a href="https://colab.research.google.com/github/3702880330101-netizen/Smart-Cost/blob/main/SmartCost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Instalar la librería oficial de Gemini si no está instalada
!pip install -q google-generativeai

import requests
import pandas as pd
import numpy as np
import google.generativeai as genai
from google.colab import userdata

# ==========================================
# 0. CONFIGURACIÓN SEGURA DE LA API DE GEMINI
# ==========================================
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=API_KEY)
    print("¡Clave de API configurada de forma segura desde los Secretos de Colab!")
except Exception as e:
    print("Error: Asegúrate de guardar tu clave en los Secretos de Colab (icono de llave) con el nombre 'GEMINI_API_KEY'.", e)

# ==========================================
# 1. CARGA AUTOMÁTICA DE DATOS DESDE GITHUB/GIST
# ==========================================
url_json = "https://gist.githubusercontent.com/3702880330101-netizen/3423b40e68604e988e434002708f03d9/raw/costos_data.json"

try:
    response = requests.get(url_json)
    response.raise_for_status()
    data = response.json()
    print("¡Datos cargados exitosamente desde tu Gist en GitHub!")
except Exception as e:
    print("Aviso: No se pudo conectar al Gist, usando valores de respaldo por defecto:", e)
    data = {
        "material": "Aluminio",
        "consumo_estandar": 1.0,
        "precio_material": 50.00,
        "desperdicio": 5.0,
        "horas_mod": 1.0,
        "tarifa_mod": 250.00,
        "cif_variable": 250.00,
        "costos_fijos": 400.00,
        "volumen": 200,
        "precio_venta": 250.00
    }

df_base = pd.DataFrame([data])
print("\n--- DATOS ACTUALES RECIBIDOS ---")
display(df_base)


# ==========================================
# 2. MOTOR DE CÁLCULO Y PUNTO DE EQUILIBRIO
# ==========================================
def calcular_modelo(p):
    material_ajustado = p['consumo_estandar'] * (1 + (p['desperdicio'] / 100))
    costo_mp_unit = material_ajustado * p['precio_material']
    costo_mod_unit = p['horas_mod'] * p['tarifa_mod']
    costo_var_unit = costo_mp_unit + costo_mod_unit + p['cif_variable']
    costo_total = (costo_var_unit * p['volumen']) + p['costos_fijos']
    costo_unitario = costo_total / p['volumen'] if p['volumen'] > 0 else 0
    margen_contrib_unit = p['precio_venta'] - costo_var_unit
    punto_equilibrio = np.ceil(p['costos_fijos'] / margen_contrib_unit) if margen_contrib_unit > 0 else 0
    utilidad = (margen_contrib_unit * p['volumen']) - p['costos_fijos']

    return {
        "Costo Unitario": costo_unitario,
        "Costo Variable Unitario": costo_var_unit,
        "Margen de Contribución": margen_contrib_unit,
        "Punto de Equilibrio (u.)": punto_equilibrio,
        "Utilidad Total": utilidad
    }

res_base = calcular_modelo(data)
print("\n--- RESULTADOS FINANCIEROS Y PUNTO DE EQUILIBRIO ---")
for k, v in res_base.items():
    print(f"• {k}: Q{v:,.2f}" if isinstance(v, float) else f"• {k}: {v}")


# ==========================================
# 3. REPORTE GERENCIAL CON GEMINI EN COLAB
# ==========================================
prompt = f"""
Actúa como un Director Financiero experto en Costos Industriales.
Analiza los siguientes resultados numéricos basados en Quetzales (Q) y redacta un informe ejecutivo centrado en el punto de equilibrio y la rentabilidad:

- Datos de entrada del producto: {data}
- Resultados Financieros y Punto de Equilibrio: {res_base}

El informe debe estructurarse con claridad e incluir:
1. Análisis del umbral de rentabilidad (Punto de Equilibrio).
2. Comportamiento de los Costos Fijos frente a los Costos Variables.
3. Recomendaciones estratégicas para optimizar el margen de contribución.
"""

try:
    model = genai.GenerativeModel("gemini-1.5-flash")
    response = model.generate_content(prompt)

    print("\n" + "=".center(60, "="))
    print(" INFORME EJECUTIVO GERENCIAL DE PUNTO DE EQUILIBRIO (IA) ".center(60, "="))
    print("=".center(60, "="))
    print(response.text)
    print("="*60)
except Exception as e:
    print("Aviso: No se pudo generar el reporte con la API de Gemini:", e)

¡Clave de API configurada de forma segura desde los Secretos de Colab!
¡Datos cargados exitosamente desde tu Gist en GitHub!

--- DATOS ACTUALES RECIBIDOS ---


,material,consumo_estandar,precio_material,desperdicio,horas_mod,tarifa_mod,cif_variable,costos_fijos,volumen,precio_venta,timestamp
0,Acero Inoxidable,1.0,50.0,5.0,2.5,15.0,10.0,5000.0,1000,120.0,2026-10-07T01:42:00.000Z



--- RESULTADOS FINANCIEROS Y PUNTO DE EQUILIBRIO ---
• Costo Unitario: Q105.00
• Costo Variable Unitario: Q100.00
• Margen de Contribución: Q20.00
• Punto de Equilibrio (u.): Q250.00
• Utilidad Total: Q15,000.00
Aviso: No se pudo generar el reporte con la API de Gemini: 401 API key not valid. Please pass a valid API key. [reason: "ACCESS_TOKEN_TYPE_UNSUPPORTED"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
metadata {
  key: "method"
  value: "google.ai.generativelanguage.v1beta.GenerativeService.GenerateContent"
}
]
